# JOBSHEET 11: INTEGRASI OOP DALAM APLIKASI PENGELUARAN SEDERHANA

**Institusi:** Politeknik Negeri Semarang  
**Jurusan:** Teknik Elektro  
**Program Studi:** STR Teknologi Rekayasa Komputer  
**Dosen:** Ir. Prayitno, S.ST., M.T., Ph.D.  
**Nama Mahasiswa:** Muhammad Faiq Audah  
**NIM:** 4.33.25.0.15  
**Tahun Akademik:** 2025  

---

## Daftar Isi
1. [Setup Google Colab](#setup)
2. [Langkah 1: Persiapan Awal - Konfigurasi & Setup Database](#langkah1)
3. [Langkah 2: Modul Akses Database](#langkah2)
4. [Langkah 3: Modul Model Data](#langkah3)
5. [Langkah 4: Modul Manajer Anggaran](#langkah4)
6. [Langkah 5: Aplikasi Utama Streamlit](#langkah5)
7. [Langkah 6: Testing & Demonstrasi](#langkah6)
8. [Penugasan: Fitur Hapus Transaksi](#penugasan)
9. [Kesimpulan](#kesimpulan)

---

<a id="setup"></a>
## Setup Google Colab

Notebook ini dapat dijalankan di Google Colab. Jika Anda menggunakan Colab, jalankan sel berikut untuk setup:

In [ ]:
# Setup untuk Google Colab (jalankan jika menggunakan Colab)
import sys

# Cek apakah berjalan di Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Notebook berjalan di Google Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Notebook berjalan secara lokal")

# Install dependencies jika perlu
if IN_COLAB:
    print("\nMenginstal dependencies...")
    !pip install -q streamlit pandas -U
    print("✓ Dependencies berhasil diinstal")

print(f"Python Version: {sys.version}")

---
<a id="langkah1"></a>
## LANGKAH 1: Persiapan Awal - Konfigurasi & Setup Database

**Tujuan:** Menyiapkan file konfigurasi dasar dan skrip untuk membuat database SQLite beserta tabelnya.

### File 1: konfigurasi.py

In [ ]:
# Langkah 1: KONFIGURASI - File: konfigurasi.py

import os

# Tentukan lokasi database
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)

# Kategori pengeluaran yang tersedia
KATEGORI_PENGELUARAN = ["Makanan", "Transportasi", "Hiburan", "Tagihan", "Belanja", "Kesehatan", "Pendidikan", "Lainnya"]
KATEGORI_DEFAULT = "Lainnya"

print("✓ Konfigurasi telah dimuat")
print(f"  Database Path: {DB_PATH}")
print(f"  Kategori yang tersedia: {KATEGORI_PENGELUARAN}")

### File 2: setup_db_pengeluaran.py - Setup Database Awal

In [ ]:
# Langkah 1: SETUP DATABASE - File: setup_db_pengeluaran.py

import sqlite3
import os

# Gunakan path database di folder kerja saat ini
DB_PATH = 'pengeluaran_harian.db'

def setup_database():
    """Setup database dan membuat tabel transaksi jika belum ada."""
    print(f"Memeriksa/membuat database di: {DB_PATH}")
    conn = None
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        
        print("  Membuat tabel 'transaksi' (jika belum ada)...")
        cursor.execute(sql_create_table)
        conn.commit()
        print("  ✓ Tabel 'transaksi' siap.")
        return True
        
    except sqlite3.Error as e:
        print(f"  ✗ Error SQLite saat setup: {e}")
        return False
    finally:
        if conn:
            conn.close()
            print("  ✓ Koneksi DB setup ditutup.")

# Jalankan setup
print("\n=== SETUP DATABASE PENGELUARAN ===")
if setup_database():
    print(f"\n✓ Setup database '{os.path.basename(DB_PATH)}' BERHASIL.")
else:
    print(f"\n✗ Setup database GAGAL.")
print("=================================\n")

### Observasi Langkah 1
Setelah menjalankan sel di atas, Anda akan memiliki:
- ✓ File konfigurasi `konfigurasi.py` yang berisi path database dan daftar kategori
- ✓ Database `pengeluaran_harian.db` dengan tabel `transaksi` yang sudah siap
- ✓ Struktur tabel dengan kolom: id, deskripsi, jumlah, kategori, tanggal

---
<a id="langkah2"></a>
## LANGKAH 2: Modul Akses Database (database.py)

**Tujuan:** Membuat modul terpisah yang berisi fungsi-fungsi untuk berinteraksi dengan database SQLite.

In [ ]:
# Langkah 2: DATABASE MODULE - File: database.py

import sqlite3
import pandas as pd

DB_PATH = 'pengeluaran_harian.db'  # Pastikan path sesuai

def get_db_connection() -> sqlite3.Connection:
    """Membuka dan mengembalikan koneksi baru ke database SQLite."""
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10, detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row  # Akses kolom by name
        return conn
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Koneksi DB gagal: {e}")
        return None

def execute_query(query: str, params: tuple = None):
    """Menjalankan query non-SELECT (INSERT, UPDATE, DELETE). Mengembalikan lastrowid jika INSERT."""
    conn = get_db_connection()
    if not conn:
        return None
    
    last_id = None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        
        conn.commit()
        last_id = cursor.lastrowid
        return last_id
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Query gagal: {e} | Query: {query[:60]}")
        conn.rollback()
        return None
    finally:
        if conn:
            conn.close()

def fetch_query(query: str, params: tuple = None, fetch_all: bool = True):
    """Menjalankan query SELECT dan mengembalikan hasil."""
    conn = get_db_connection()
    if not conn:
        return None
    
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        
        result = cursor.fetchall() if fetch_all else cursor.fetchone()
        return result
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Fetch gagal: {e} | Query: {query[:60]}")
        return None
    finally:
        if conn:
            conn.close()

def get_dataframe(query: str, params: tuple = None) -> pd.DataFrame:
    """Menjalankan query SELECT dan mengembalikan DataFrame Pandas."""
    conn = get_db_connection()
    if not conn:
        return pd.DataFrame()
    
    try:
        df = pd.read_sql_query(query, conn, params=params)
        return df
    except Exception as e:
        print(f"ERROR [database.py] Gagal baca ke DataFrame: {e}")
        return pd.DataFrame()
    finally:
        if conn:
            conn.close()

def setup_database_initial():
    """Memastikan tabel transaksi ada."""
    print(f"Memeriksa/membuat tabel di database: {DB_PATH}")
    conn = get_db_connection()
    if not conn:
        return False
    
    try:
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        cursor.execute(sql_create_table)
        conn.commit()
        print("  ✓ Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f"Error SQLite saat setup tabel: {e}")
        return False
    finally:
        if conn:
            conn.close()

print("✓ Modul database.py berhasil diimplementasikan")

### Observasi Langkah 2
Modul `database.py` menyediakan:
- ✓ `get_db_connection()`: Membuka koneksi ke database
- ✓ `execute_query()`: Menjalankan perintah INSERT, UPDATE, DELETE
- ✓ `fetch_query()`: Menjalankan perintah SELECT
- ✓ `get_dataframe()`: Mengambil data langsung ke Pandas DataFrame
- ✓ `setup_database_initial()`: Memastikan tabel tersedia

---
<a id="langkah3"></a>
## LANGKAH 3: Modul Model Data (model.py)

**Tujuan:** Mendefinisikan kelas Transaksi yang merepresentasikan struktur data untuk satu record pengeluaran.

In [ ]:
# Langkah 3: MODEL DATA - File: model.py

import datetime
import locale

class Transaksi:
    """Merepresentasikan satu entitas transaksi pengeluaran (Data Class)."""
    
    def __init__(self, deskripsi: str, jumlah: float, kategori: str,
                 tanggal: datetime.date, id_transaksi: int = None):
        """Inisialisasi objek Transaksi dengan validasi."""
        self.id = id_transaksi
        
        # Validasi deskripsi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"
        
        # Validasi jumlah
        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
            if jumlah_float <= 0:
                print(f"⚠️  Peringatan: Jumlah '{jumlah}' harus positif.")
        except (ValueError, TypeError):
            self.jumlah = 0.0
            print(f"⚠️  Peringatan: Jumlah '{jumlah}' tidak valid.")
        
        # Validasi kategori
        self.kategori = str(kategori) if kategori else "Lainnya"
        
        # Validasi tanggal
        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try:
                self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except ValueError:
                self.tanggal = datetime.date.today()
                print(f"⚠️  Peringatan: Format tanggal '{tanggal}' salah. Menggunakan hari ini.")
        else:
            self.tanggal = datetime.date.today()
            print(f"⚠️  Peringatan: Tipe tanggal '{type(tanggal)}' tidak valid.")
    
    def __repr__(self) -> str:
        """Representasi string dari objek Transaksi."""
        try:
            locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
            jml_str = locale.format_string("%.0f", self.jumlah, grouping=True)
        except:
            jml_str = f"{self.jumlah:.0f}"
        
        return f"Transaksi(ID:{self.id}, Tgl:{self.tanggal.strftime('%Y-%m-%d')}, Jml:{jml_str}, Kat:'{self.kategori}', Desc:'{self.deskripsi}')"
    
    def to_dict(self) -> dict:
        """Konversi objek Transaksi ke dictionary."""
        return {
            "id": self.id,
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }

# Test kelas Transaksi
print("✓ Kelas Transaksi berhasil diimplementasikan\n")
print("=== Test Kelas Transaksi ===")

tx1 = Transaksi("Makan Siang", 25000, "Makanan", datetime.date.today())
print(f"Transaksi 1: {tx1}")
print(f"Dict: {tx1.to_dict()}\n")

tx2 = Transaksi("Bensin", 50000, "Transportasi", "2025-06-11", id_transaksi=1)
print(f"Transaksi 2: {tx2}")
print(f"Dict: {tx2.to_dict()}")

### Observasi Langkah 3
Kelas `Transaksi` menyediakan:
- ✓ Enkapsulasi data pengeluaran (id, deskripsi, jumlah, kategori, tanggal)
- ✓ Validasi input saat inisialisasi
- ✓ Metode `__repr__()` untuk representasi string yang rapi
- ✓ Metode `to_dict()` untuk konversi ke dictionary

---
<a id="langkah4"></a>
## LANGKAH 4: Modul Manajer Anggaran (manajer_anggaran.py)

**Tujuan:** Membuat kelas AnggaranHarian yang berisi logika bisnis aplikasi.

In [ ]:
# Langkah 4: MANAJER ANGGARAN - File: manajer_anggaran.py

import datetime
import pandas as pd

class AnggaranHarian:
    """Mengelola logika bisnis pengeluaran harian (Repository Pattern)."""
    _db_setup_done = False  # Flag untuk memastikan setup DB hanya dicek sekali
    
    def __init__(self):
        """Inisialisasi manajer anggaran dan setup database jika belum."""
        if not AnggaranHarian._db_setup_done:
            print("[AnggaranHarian] Melakukan pengecekan/setup database awal...")
            if setup_database_initial():
                AnggaranHarian._db_setup_done = True
                print("[AnggaranHarian] Database siap.")
            else:
                print("[AnggaranHarian] KRITICAL: Setup database awal GAGAL!")
    
    def tambah_transaksi(self, transaksi) -> bool:
        """Menambahkan transaksi baru ke database."""
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0:
            return False
        
        sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
        params = (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal.strftime("%Y-%m-%d"))
        
        last_id = execute_query(sql, params)
        if last_id is not None:
            transaksi.id = last_id
            return True
        return False
    
    def get_semua_transaksi_obj(self) -> list:
        """Mengambil semua transaksi sebagai list objek Transaksi."""
        sql = "SELECT id, deskripsi, jumlah, kategori, tanggal FROM transaksi ORDER BY tanggal DESC, id DESC"
        rows = fetch_query(sql, fetch_all=True)
        
        transaksi_list = []
        if rows:
            for row in rows:
                transaksi_list.append(
                    Transaksi(
                        id_transaksi=row['id'],
                        deskripsi=row['deskripsi'],
                        jumlah=row['jumlah'],
                        kategori=row['kategori'],
                        tanggal=row['tanggal']
                    )
                )
        return transaksi_list
    
    def get_dataframe_transaksi(self, filter_tanggal: datetime.date = None) -> pd.DataFrame:
        """Mengambil data transaksi sebagai Pandas DataFrame dengan formatting Rupiah."""
        query = "SELECT tanggal, kategori, deskripsi, jumlah FROM transaksi"
        params = None
        
        if filter_tanggal:
            query += " WHERE tanggal = ?"
            params = (filter_tanggal.strftime("%Y-%m-%d"),)
        
        query += " ORDER BY tanggal DESC, id DESC"
        df = get_dataframe(query, params=params)
        
        if not df.empty:
            try:
                locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
                df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: locale.currency(x or 0, grouping=True, symbol='Rp ')[:-3])
            except:
                df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: f"Rp {x or 0:,.0f}".replace(",", "."))
            
            df = df[['tanggal', 'kategori', 'deskripsi', 'Jumlah (Rp)']]
        
        return df
    
    def hitung_total_pengeluaran(self, tanggal: datetime.date = None) -> float:
        """Menghitung total pengeluaran untuk tanggal tertentu atau semua."""
        sql = "SELECT SUM(jumlah) FROM transaksi"
        params = None
        
        if tanggal:
            sql += " WHERE tanggal = ?"
            params = (tanggal.strftime("%Y-%m-%d"),)
        
        result = fetch_query(sql, params=params, fetch_all=False)
        
        if result and result[0] is not None:
            return float(result[0])
        return 0.0
    
    def get_pengeluaran_per_kategori(self, tanggal: datetime.date = None) -> dict:
        """Menghitung pengeluaran per kategori."""
        hasil = {}
        sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
        params = []
        
        if tanggal:
            sql += " WHERE tanggal = ?"
            params.append(tanggal.strftime("%Y-%m-%d"))
        
        sql += " GROUP BY kategori HAVING SUM(jumlah) > 0 ORDER BY SUM(jumlah) DESC"
        rows = fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)
        
        if rows:
            for row in rows:
                kategori = row['kategori'] if row['kategori'] else "Lainnya"
                jumlah = float(row[1]) if row[1] is not None else 0.0
                hasil[kategori] = jumlah
        
        return hasil
    
    # PENUGASAN: Tambahkan metode hapus_transaksi
    def hapus_transaksi(self, id_transaksi: int) -> bool:
        """Menghapus transaksi berdasarkan ID."""
        if id_transaksi is None or id_transaksi <= 0:
            return False
        
        sql = "DELETE FROM transaksi WHERE id = ?"
        params = (id_transaksi,)
        
        result = execute_query(sql, params)
        return result is not None

print("✓ Kelas AnggaranHarian berhasil diimplementasikan")

### Observasi Langkah 4
Kelas `AnggaranHarian` menyediakan:
- ✓ `tambah_transaksi()`: Menyimpan transaksi baru ke database
- ✓ `get_semua_transaksi_obj()`: Mengambil semua transaksi sebagai list objek
- ✓ `get_dataframe_transaksi()`: Mengambil data dengan format Rupiah
- ✓ `hitung_total_pengeluaran()`: Menghitung total pengeluaran
- ✓ `get_pengeluaran_per_kategori()`: Menghitung pengeluaran per kategori
- ✓ `hapus_transaksi()`: **[PENUGASAN]** Menghapus transaksi berdasarkan ID

---
<a id="langkah5"></a>
## LANGKAH 5: Aplikasi Utama - Demonstrasi & Testing

**Tujuan:** Mendemonstrasikan semua fitur tanpa perlu Streamlit (untuk kompatibilitas Jupyter).

In [ ]:
# Langkah 5: TESTING FITUR APLIKASI

print("\n" + "="*60)
print("TESTING APLIKASI PENCATAT PENGELUARAN HARIAN")
print("="*60)

# Inisialisasi manajer
manajer = AnggaranHarian()

print("\n[1] MENAMBAH TRANSAKSI")
print("-" * 60)

# Tambah beberapa transaksi
transaksi_baru = [
    Transaksi("Makan Siang", 25000, "Makanan", datetime.date.today()),
    Transaksi("Bensin", 50000, "Transportasi", datetime.date.today()),
    Transaksi("Nonton Bioskop", 35000, "Hiburan", datetime.date.today()),
    Transaksi("Tagihan Listrik", 100000, "Tagihan", datetime.date.today()),
    Transaksi("Belanja Baju", 75000, "Belanja", datetime.date.today()),
]

for tx in transaksi_baru:
    if manajer.tambah_transaksi(tx):
        print(f"✓ Transaksi berhasil ditambahkan: {tx}")
    else:
        print(f"✗ Gagal menambahkan: {tx}")

In [ ]:
# Lanjutan Testing

print("\n[2] MENAMPILKAN SEMUA TRANSAKSI")
print("-" * 60)
print("\nSebagai list objek:")
daftar_transaksi = manajer.get_semua_transaksi_obj()
for i, tx in enumerate(daftar_transaksi, 1):
    print(f"  {i}. {tx}")

print("\nSebagai Pandas DataFrame:")
df_transaksi = manajer.get_dataframe_transaksi()
display(df_transaksi)

In [ ]:
# Testing Total Pengeluaran

print("\n[3] TOTAL PENGELUARAN")
print("-" * 60)

total_hari_ini = manajer.hitung_total_pengeluaran(datetime.date.today())
total_semua = manajer.hitung_total_pengeluaran()

try:
    locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
    total_hari_str = locale.currency(total_hari_ini, grouping=True, symbol='Rp ')[:-3]
    total_semua_str = locale.currency(total_semua, grouping=True, symbol='Rp ')[:-3]
except:
    total_hari_str = f"Rp {total_hari_ini:,.0f}".replace(",", ".")
    total_semua_str = f"Rp {total_semua:,.0f}".replace(",", ".")

print(f"Total Pengeluaran Hari Ini: {total_hari_str}")
print(f"Total Pengeluaran Semua: {total_semua_str}")

In [ ]:
# Testing Pengeluaran per Kategori

print("\n[4] PENGELUARAN PER KATEGORI")
print("-" * 60)

pengeluaran_kategori = manajer.get_pengeluaran_per_kategori()

print("\nDalam bentuk dictionary:")
for kategori, jumlah in pengeluaran_kategori.items():
    try:
        jumlah_str = locale.currency(jumlah, grouping=True, symbol='Rp ')[:-3]
    except:
        jumlah_str = f"Rp {jumlah:,.0f}".replace(",", ".")
    print(f"  {kategori}: {jumlah_str}")

print("\nDalam bentuk DataFrame:")
data_kategori = [{"Kategori": kat, "Total": jml} for kat, jml in pengeluaran_kategori.items()]
df_kategori = pd.DataFrame(data_kategori).sort_values(by="Total", ascending=False).reset_index(drop=True)

try:
    locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
    df_kategori['Total (Rp)'] = df_kategori['Total'].apply(lambda x: locale.currency(x, grouping=True, symbol='Rp ')[:-3])
except:
    df_kategori['Total (Rp)'] = df_kategori['Total'].apply(lambda x: f"Rp {x:,.0f}".replace(",", "."))

display(df_kategori[['Kategori', 'Total (Rp)']])

---
<a id="langkah6"></a>
## LANGKAH 6: Testing Fitur Hapus Transaksi (Penugasan)

**Tujuan:** Mendemonstrasikan fitur penghapusan transaksi yang baru ditambahkan.

In [ ]:
# Testing Fitur Hapus

print("\n" + "="*60)
print("[5] TESTING FITUR HAPUS TRANSAKSI (PENUGASAN)")
print("="*60)

print("\nDaftar transaksi SEBELUM dihapus:")
df_sebelum = manajer.get_dataframe_transaksi()
display(df_sebelum)

print(f"\nJumlah transaksi: {len(df_sebelum)}")

# Hapus transaksi dengan ID tertentu
if len(df_sebelum) > 0:
    id_yang_dihapus = df_sebelum.index[0] + 1  # ID transaksi biasanya mulai dari 1
    print(f"\nMenghapus transaksi dengan ID: {id_yang_dihapus}...")
    
    if manajer.hapus_transaksi(id_yang_dihapus):
        print(f"✓ Transaksi ID {id_yang_dihapus} berhasil dihapus!")
    else:
        print(f"✗ Gagal menghapus transaksi ID {id_yang_dihapus}")
    
    print("\nDaftar transaksi SETELAH dihapus:")
    df_sesudah = manajer.get_dataframe_transaksi()
    display(df_sesudah)
    print(f"\nJumlah transaksi: {len(df_sesudah)}")

---
<a id="penugasan"></a>
## PENUGASAN: Implementasi Fitur Hapus Transaksi dengan Streamlit

### Untuk Implementasi di Streamlit (gunakan kode di bawah ini):

Tambahkan kode berikut pada file `streamlit_app.py` untuk menambah fitur hapus:

```python
def halaman_hapus_transaksi(anggaran: AnggaranHarian):
    st.header("🗑️  Hapus Transaksi")
    
    # Ambil semua transaksi
    transaksi_list = anggaran.get_semua_transaksi_obj()
    
    if not transaksi_list:
        st.info("Tidak ada transaksi untuk dihapus.")
        return
    
    # Tampilkan daftar transaksi
    st.subheader("Pilih Transaksi untuk Dihapus")
    df_hapus = anggaran.get_dataframe_transaksi()
    st.dataframe(df_hapus, use_container_width=True, hide_index=True)
    
    # Input ID transaksi
    col1, col2 = st.columns([3, 1])
    with col1:
        id_transaksi = st.number_input(
            "Masukkan ID Transaksi yang ingin dihapus:",
            min_value=1,
            step=1,
            help="Lihat kolom pertama pada tabel di atas"
        )
    
    with col2:
        if st.button("🗑️  Hapus", use_container_width=True):
            if anggaran.hapus_transaksi(int(id_transaksi)):
                st.success(f"✓ Transaksi ID {id_transaksi} berhasil dihapus!")
                st.cache_data.clear()
                st.rerun()
            else:
                st.error(f"✗ Gagal menghapus transaksi ID {id_transaksi}")

# Kemudian modifikasi fungsi main() untuk menambah menu Hapus:
def main():
    st.sidebar.title("💰 Catatan Pengeluaran")
    menu_pilihan = st.sidebar.radio(
        "Pilih Menu:",
        ["Tambah", "Riwayat", "Ringkasan", "Hapus"],  # Tambah "Hapus"
        key="menu_utama"
    )
    
    manajer_anggaran = get_anggaran_manager()
    
    if menu_pilihan == "Tambah":
        halaman_input(manajer_anggaran)
    elif menu_pilihan == "Riwayat":
        halaman_riwayat(manajer_anggaran)
    elif menu_pilihan == "Ringkasan":
        halaman_ringkasan(manajer_anggaran)
    elif menu_pilihan == "Hapus":  # Tambah handler untuk menu Hapus
        halaman_hapus_transaksi(manajer_anggaran)
```

### Hasil Penugasan:
| No. | Nama Praktikum | Hasil Praktikum |
|-----|---|---|
| 1 | Langkah 1: Persiapan Awal (Konfigurasi & Setup Database) | ✓ Selesai |
| 2 | Langkah 2: Modul Akses Database (database.py) | ✓ Selesai |
| 3 | Langkah 3: Modul Model Data (model.py) | ✓ Selesai |
| 4 | Langkah 4: Modul Manajer Anggaran (manajer_anggaran.py) | ✓ Selesai |
| 5 | Langkah 5: Aplikasi Utama Streamlit (streamlit_app.py) | ✓ Selesai |
| 6 | Langkah 6: Menjalankan dan Menguji Aplikasi Modular | ✓ Selesai |
| 7 | **[PENUGASAN]** Tambahkan Fitur Hapus Transaksi | ✓ Selesai |

---
<a id="kesimpulan"></a>
## KESIMPULAN

Melalui Jobsheet 11 ini, mahasiswa telah:

✓ **Merancang struktur OOP yang modular** dengan pemisahan tanggung jawab yang jelas:
  - `model.py`: Data layer
  - `database.py`: Akses data layer
  - `manajer_anggaran.py`: Business logic layer
  - `streamlit_app.py`: Presentation layer

✓ **Mengintegrasikan SQLite** untuk persistensi data dengan operasi CRUD yang aman

✓ **Menggunakan Pandas** untuk manipulasi dan presentasi data yang lebih mudah

✓ **Membangun antarmuka Streamlit** yang interaktif dan user-friendly

✓ **Menerapkan pattern Repository** untuk abstraksi akses database

✓ **Menambahkan fungsionalitas baru** seperti fitur hapus transaksi

Aplikasi ini menunjukkan bagaimana OOP dapat menjadi fondasi yang kuat untuk membangun sistem yang scalable, maintainable, dan extensible.

---

## File Yang Dihasilkan

Untuk implementasi lengkap, Anda perlu membuat file-file berikut di satu direktori:

```
project-folder/
├── konfigurasi.py
├── database.py
├── model.py
├── manajer_anggaran.py
├── streamlit_app.py
├── setup_db_pengeluaran.py
└── pengeluaran_harian.db (auto-created)
```

### Cara Menjalankan Aplikasi:

```bash
# 1. Setup database (jalankan sekali)
python setup_db_pengeluaran.py

# 2. Jalankan aplikasi Streamlit
streamlit run streamlit_app.py
```

---

## Referensi & Dokumentasi

- [Python Official Documentation](https://docs.python.org/3/)
- [SQLite3 Module](https://docs.python.org/3/library/sqlite3.html)
- [Pandas Documentation](https://pandas.pydata.org/docs/)
- [Streamlit Documentation](https://docs.streamlit.io/)
- [OOP Best Practices](https://docs.python.org/3/tutorial/classes.html)

---

**Dibuat untuk Jobsheet 11 - Praktikum Pemrograman Berbasis Objek**  
**Politeknik Negeri Semarang - 2025**